# 01 - Single Agent vs Multi-Agent Architectures

## Scenario: Triaging Complex Network Incidents

Many beginners try to build a "God Agent" by stuffing 50 tools into a single System Prompt. 

As tasks become complex, a single agent will suffer from **Context Dilution** and **Tool Confusion**. In this module, we will compare:
1. **The Monolithic Agent**: A single agent with DBA, Network, and Support tools.
2. **The Multi-Agent Orchestration**: A Triage agent that routes tasks to specialized sub-agents (DBA Agent, Network Agent).

In [1]:
# 1. Initialization and Mock Fallback
import os
import sys

# Attempt to use real API key
if os.environ.get("OPENAI_API_KEY"):
    from openai import OpenAI
    client = OpenAI()
    print("✅ Using real OpenAI API.")
else:
    print("⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...")
    # Add repo root to path for local execution without pip install
    sys.path.append(os.path.abspath("../../.."))
    try:
        from awsome_agents.mock_openai import MockOpenAI
        client = MockOpenAI()
    except ImportError:
        print("Failed to import MockOpenAI. Ensure you are running from the repository root.")

⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...
🔧 Initialized MockOpenAI Client (Network requests disabled)


## 1. The Monolithic "God" Agent

Notice how complex the system prompt gets when we try to cover every scenario. The agent often hallucinates or uses the wrong tool because its attention is split.

In [2]:
monolithic_prompt = """
You are the Northstar God Agent. You handle all incidents.
- If it's a database issue, use the `check_db_locks` tool.
- If it's a network issue, use the `ping_vpc` tool.
- If it's a billing issue, use the `refund` tool.
- If the user is angry, apologize. If it's a server issue, do NOT apologize.
- If latency > 500ms, scale up the DB. If ping fails, restart the router.
"""

# The agent gets confused easily when multiple conditions overlap
print("Monolithic Prompt Length:", len(monolithic_prompt.split()))


Monolithic Prompt Length: 70


## 2. Multi-Agent Orchestration

Instead of one massive prompt, we build a **Router/Triage Agent**. Its ONLY job is to classify the issue and hand it off to a specialized agent.

In [3]:
def network_agent(incident: str) -> str:
    print("  🌐 [Network Agent] Taking over...")
    print("  🌐 [Network Agent] Executing `ping_vpc`...")
    return "Network is healthy."

def dba_agent(incident: str) -> str:
    print("  🗄️ [DBA Agent] Taking over...")
    print("  🗄️ [DBA Agent] Executing `check_db_locks`...")
    return "Deadlock found on table 'orders'. Cleared."

def triage_agent(incident: str):
    print(f"🚦 [Triage Agent] Analyzing incident: '{incident}'")
    
    # In a real system, the Triage agent uses Structured Outputs to route the request
    if "timeout" in incident.lower() or "latency" in incident.lower():
        result = dba_agent(incident)
    elif "unreachable" in incident.lower():
        result = network_agent(incident)
    else:
        result = "Unknown issue. Escalating to human."
        
    print(f"🚦 [Triage Agent] Final Resolution: {result}")

triage_agent("We are seeing a massive latency spike on the checkout page.")


🚦 [Triage Agent] Analyzing incident: 'We are seeing a massive latency spike on the checkout page.'
  🗄️ [DBA Agent] Taking over...
  🗄️ [DBA Agent] Executing `check_db_locks`...
🚦 [Triage Agent] Final Resolution: Deadlock found on table 'orders'. Cleared.


## Checkpoint

**1. Why is a Multi-Agent architecture preferred over a single "God Agent" for complex systems?**
- A) It reduces the total number of API calls.
- B) It allows you to enforce specialized personas, restrict tool access (Principle of Least Privilege), and prevent prompt dilution.
- C) It is faster to execute.
- D) It bypasses OpenAI rate limits.
